# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [21]:
from dotenv import load_dotenv
import os

load_dotenv()

price_data_path = os.getenv("PRICE_DATA")
print(price_data_path)

../../05_src/data/prices/


In [22]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [23]:
import os
from glob import glob

parquet_files = glob(os.path.join(price_data_path, "**", "*.parquet"), recursive=True)

print(parquet_files[:3])


['../../05_src/data/prices/BKTI/BKTI_2012/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2012/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2015/part.0.parquet']


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [24]:
dask_df=dd.read_parquet(parquet_files)


dask_df.head()



,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year
38739,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,A,1999
38740,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,A,1999
38741,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,A,1999
38742,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,A,1999
38743,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,A,1999


In [26]:

dask_df['Close_Lag_1'] = dask_df.groupby('ticker')['Close'].shift(1)

dask_df['Adj_Close_Lag_1'] = dask_df.groupby('ticker')['Adj Close'].shift(1)

dask_df['returns'] = (dask_df['Close']/dask_df['Close_Lag_1'])-1

dask_df['hi_lo_range'] = dask_df['High'] - dask_df['Low']

dd_feat = dask_df


/var/folders/04/z3jvs4bd4c1bl816136pyshc0000gn/T/ipykernel_30047/3725161804.py:1: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dask_df['Close_Lag_1'] = dask_df.groupby('ticker')['Close'].shift(1)
/var/folders/04/z3jvs4bd4c1bl816136pyshc0000gn/T/ipykernel_30047/3725161804.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dask_df['Adj_Close_Lag_1'] = dask_df.groupby('ticker')['Adj Close'].shift(1)


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [27]:
feat_df = dd_feat.compute()


feat_df['returns_moving_avg_10'] = (feat_df.groupby('ticker')['returns']
                                    .transform(lambda x: x.rolling(10).mean())
)

KeyboardInterrupt: 

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was not necessary since the dataset is big so using Dask is more efficient.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.